# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [3]:
# Task 1
# YOUR CODE HERE
# Task 1
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Same colours are reused in Task 3 (Okabe-Ito, colour-blind safe)
source_colors = {
    'Coal': '#D55E00',
    'Oil': '#0072B2',
    'Natural Gas': '#F0E442',
    '(?)': '#D9D9D9'          # parent nodes (Region, Country) in grey
}

# Insight title from the data
top_region = df_fossil.groupby('Region')['TWh'].sum().idxmax()
top_region_twh = df_fossil.groupby('Region')['TWh'].sum().max()
top_country = df_fossil.groupby('Country')['TWh'].sum().idxmax()
top_country_twh = df_fossil.groupby('Country')['TWh'].sum().max()

fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=source_colors,
    title=(f"{top_region} is the most fossil-dependent region "
           f"({top_region_twh:,.0f} TWh), led by {top_country} "
           f"({top_country_twh:,.0f} TWh)")
)
fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
    textfont_size=13
)
fig1.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig1.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
# YOUR CODE HERE
# Task 2 - charts
df_low = (df.loc[df['Source_Type'] == 'Low-carbon']
            .groupby('Country', as_index=False)['TWh'].sum()
            .sort_values('TWh', ascending=False))
df_low['All'] = 'Low-carbon'

low_blue = '#0072B2'

# Treemap
fig2a = px.treemap(
    df_low,
    path=['All', 'Country'],
    values='TWh',
    color='All',
    color_discrete_map={'Low-carbon': low_blue, '(?)': '#D9D9D9'},
    title='Low-carbon (Nuclear + Hydro) electricity by country, TWh'
)
fig2a.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>'
)
fig2a.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig2a.show()

# Horizontal bar, sorted so the largest is at the top
df_low_bar = df_low.sort_values('TWh', ascending=True)
leader = df_low.iloc[0]

fig2b = px.bar(
    df_low_bar,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=[low_blue],
    title=f"{leader['Country']} leads low-carbon energy with {leader['TWh']:,.0f} TWh"
)
fig2b.update_traces(texttemplate='%{x:,.0f} TWh', textposition='outside')
fig2b.update_layout(
    xaxis_title='Low-carbon energy (TWh)',
    yaxis_title=None,
    plot_bgcolor='white',
    margin=dict(t=60, r=80)
)
fig2b.update_xaxes(range=[0, df_low['TWh'].max() * 1.15])
fig2b.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

The bar chart tells the ranking story more clearly. Bars share a common baseline, so the eye compares length, which is the most accurate visual comparison. The countries are sorted, so the order is immediately readable. In the treemap, rank has to be judged from rectangle areas with different shapes, which is harder, especially for countries with similar values. The treemap is better at showing each country's share of the total, but the question here is "who is first, second, third", and the bar chart answers that.


## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
# YOUR CODE HERE
# Task 3
fossil_region = (df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum())
fossil_region['Share_pct'] = (
    fossil_region['TWh'] / fossil_region.groupby('Region')['TWh'].transform('sum') * 100
)

# Order regions by coal share. Horizontal bars plot the first category at the
# bottom, so sort ascending to get the highest coal share at the top.
coal_share = (fossil_region.loc[fossil_region['Source'] == 'Coal']
              .set_index('Region')['Share_pct']
              .reindex(fossil_region['Region'].unique())
              .fillna(0)
              .sort_values())
region_order = coal_share.index.tolist()

top_coal_region = coal_share.idxmax()
top_coal_share = coal_share.max()

fig3 = px.bar(
    fossil_region,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    category_orders={'Source': ['Coal', 'Oil', 'Natural Gas'],
                     'Region': region_order},
    color_discrete_map={'Coal': source_colors['Coal'],
                        'Oil': '#8C8C8C',
                        'Natural Gas': '#CFCFCF'},
    title=f"{top_coal_region} relies most on coal: {top_coal_share:.0f}% of its fossil energy"
)

# Percentage labels on coal only
fig3.update_traces(texttemplate='%{x:.0f}%', textposition='inside',
                   textfont_color='white', selector=dict(name='Coal'))
fig3.update_traces(texttemplate='', selector=dict(name='Oil'))
fig3.update_traces(texttemplate='', selector=dict(name='Natural Gas'))

fig3.update_layout(
    xaxis_title='Share of regional fossil energy (%)',
    yaxis_title=None,
    xaxis=dict(range=[0, 100]),
    legend_title_text='',
    plot_bgcolor='white',
    margin=dict(t=60)
)
fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*

The treemap answers absolute questions, for example "which country burns the most fossil fuel in TWh?" or "how big is Asia's fossil use compared to Europe's?". The stacked bar cannot show this, because every region is scaled to 100% and the size of the region disappears. The stacked bar answers the composition question, for example "which region depends most on coal relative to its own fossil mix?". The treemap cannot show this clearly, because a small region with a high coal share just looks like a small rectangle.